# GradeSight — Fine-tune Qwen2.5-VL-3B for exam extract

Colab / GPU: **preprocess → load → LoRA train → evaluate → export**.

After export, serve locally and point Next.js at it:
```bash
uvicorn ml.serve_extract:app --host 127.0.0.1 --port 8001
# .env.local: LOCAL_EXTRACT_URL=http://127.0.0.1:8001
```

In [ ]:
# 0) Setup (Colab: mount drive / clone repo first)
%pip install -q -r ../requirements.txt
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.makedirs('../artifacts', exist_ok=True)

In [ ]:
# 1) Preprocess — handwriting OCR + fixture synthetic samples
from preprocess import build_dataset
from datasets import Dataset

ds = build_dataset(max_handwriting=80, seed=42)
out = '../artifacts/dataset'
ds.save_to_disk(out)
print(len(ds), 'samples →', out)

In [ ]:
# 2) Load Qwen2.5-VL-3B + LoRA
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    trust_remote_code=True,
)
lora = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 3) Fine-tune (short Colab run — raise max_steps for quality)
!python ../train.py --dataset ../artifacts/dataset --out ../artifacts/adapter --max-steps 60 --epochs 1

In [ ]:
# 4) Evaluation metrics (CER/WER, label F1, bbox IoU)
!python ../evaluate.py --demo --out ../artifacts/metrics.json
import json
print(json.load(open('../artifacts/metrics.json')))

## 5) Export

Adapter lives at `ml/artifacts/adapter`. Download from Colab, then:
```bash
EXTRACT_ADAPTER_PATH=ml/artifacts/adapter python -m ml.serve_extract
```